# 06 - Prepare Per-Crop Disease Datasets

Stage 6: Build per-crop disease datasets from manifest.csv (produced by
01_prepare_crop_dataset.py, which already recorded disease_raw + environment
per image).

WHY THIS NEEDS YOUR INPUT, NOT JUST A RUN:
Your raw folder names use inconsistent taxonomies across environments (e.g.
Potato Closed uses "Early Blight"/"Late Blight" while Potato Uncontrolled
uses broad categories like "Fungi"/"Bacteria" -- these are NOT the same
labeling system and should not be blindly merged). CANONICAL_DISEASE_MAP
below is a STARTER mapping fixing only unambiguous typos/casing. Review
and extend it yourself, crop by crop, before trusting the output --
especially the flagged crops.

Install deps:
    pip install pandas scikit-learn tqdm --break-system-packages

## Imports & Configuration

In [13]:
import shutil
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

MANIFEST_PATH = Path("manifest.csv")
OUT_DIR = Path("disease_dataset")
MIN_CLASS_COUNT = 30  # classes with fewer images than this are dropped, not trained on

# ---- Pest-damage folders excluded from disease classification -----------
# Confirmed: these are symptoms of pests, not diseases. Excluded from
# training now -- reserved for the future YOLOv8 pest-detection module.
# REMINDER (as requested): don't lose track of these -- they still need a
# home in the Pest Detection Service down the line.
PEST_CLASSES_TO_EXCLUDE = {
    "Aphids", "Aphid", "Army worm", "Spider mite", "Spider Mite",
    "Thrips", "Leaf Miner", "Leaf miners", "Leaf miner",
}

# ---- Canonical mapping: crop -> {raw_label: canonical_label} ------------
# Confirmed final taxonomy per crop. Correct-spelling canonical names only.
# Pest-symptom rows above are still mapped here (for consistent spelling)
# but get filtered out by PEST_CLASSES_TO_EXCLUDE before training.
CANONICAL_DISEASE_MAP = {
    "Cotton": {
        "Alternaria Leaf Spot": "Alternaria Leaf Spot",
        "Bacterial Blight": "Bacterial Blight",
        "Fusarium Wilt": "Fusarium Wilt",
        "Fussarium Wilt": "Fusarium Wilt",  # typo fix
        "Healthy Leaf": "Healthy",
        "healthy": "Healthy",
        "Verticillium Wilt": "Verticillium Wilt",
        "Curl Virus": "Curl Virus",
        "Powdery Mildew": "Powdery Mildew",
        "Target spot": "Target Spot",
        "Aphids": "Aphids",          # excluded (pest symptom)
        "Army worm": "Army Worm",    # excluded (pest symptom)
    },
    "Groundnut": {
        # Confirmed: early + late leaf spot combined into one "Leaf Spot"
        # class (including Field Closeup's already-merged folder and
        # early_rust, which folds into "Rust" since no separate
        # "Early Rust" class was requested).
        "early_leaf_spot_1": "Leaf Spot",
        "early_leaf_spot": "Leaf Spot",
        "late_leaf_spot_1": "Leaf Spot",
        "late leaf spot": "Leaf Spot",
        "LEAF SPOT (EARLY AND LATE)": "Leaf Spot",
        "ALTERNARIA LEAF SPOT": "Alternaria Leaf Spot",
        "ROSETTE": "Rosette",
        "healthy_leaf_1": "Healthy",
        "healthy leaf": "Healthy",
        "HEALTHY": "Healthy",
        "nutrition_deficiency_1": "Nutrition Deficiency",
        "nutrition deficiency": "Nutrition Deficiency",
        "rust_1": "Rust",
        "rust": "Rust",
        "RUST": "Rust",
        "early_rust_1": "Rust",
    },
    "Pepper Bell": {
        "Bacterial Spot": "Bacterial Spot",
        "Bacterial spot": "Bacterial Spot",
        "Cercospora Leaf Spot": "Cercospora Leaf Spot",
        "Healthy": "Healthy",
        "Leaf Curl": "Leaf Curl",
        "Leaf curl": "Leaf Curl",
        "Nutrition Deficiency": "Nutrient Deficiency",
        "Nutrient deficiency": "Nutrient Deficiency",
        "Powdery Mildew": "Powdery Mildew",
        "Powdery mildew": "Powdery Mildew",
        "Blossom end rot": "Blossom End Rot",
        # Kept as its own class per your note that "Burn" is often
        # mislabeled/grouped with Blossom End Rot or Nutrient Deficiency
        # depending on the dataset creator -- only 5 images, so
        # MIN_CLASS_COUNT will prune it automatically unless you add more.
        "Burn": "Burn",
        "Edema": "Edema",
        "Aphid": "Aphid",              # excluded (pest symptom)
        "Leaf miners": "Leaf Miners",   # excluded (pest symptom)
        "Spider mite": "Spider Mite",   # excluded (pest symptom)
        "Thrips": "Thrips",             # excluded (pest symptom)
        # NOTE: "Mosaic Virus" was on your confirmed list but there's no
        # matching raw folder in your current Pepper Bell directories --
        # if you have images for it, add the raw folder name here.
    },
    "Potato": {
        # Confirmed: keep Closed Environment's specific names AND
        # Uncontrolled's broad categories as their own separate classes
        # (no forced merge), except Late Blight = Phytopthora per your
        # instruction.
        "Early Blight": "Early Blight",
        "Late Blight": "Late Blight",
        "Phytopthora": "Late Blight",   # merged per your instruction
        "Healthy": "Healthy",
        "Bacteria": "Bacteria",
        "Fungi": "Fungi",
        "Virus": "Virus",
        "Nematode": "Nematode",
        "Pest": "Pest",
    },
    "Tomato": {
        "Bacterial Spot": "Bacterial Spot",
        "Early Blight": "Early Blight",
        "Late Blight": "Late Blight",
        "Healthy": "Healthy",
        "Mold Leaf": "Mold Leaf",
        "Mosaic Virus": "Mosaic Virus",
        "Septoria": "Septoria",
        "Yellow Curl Virus": "Yellow Curl Virus",
        "Defiency of Phosphor Magnesium": "Deficiency of Phosphorus and Magnesium",
        "Burn Leaf": "Burn Leaf",
        "Wealth Leaf": "Wealth Leaf",   # unclear label -- verify what this actually means
        "Leaf Miner": "Leaf Miner",     # excluded (pest symptom)
        "Spider Mite": "Spider Mite",   # excluded (pest symptom)
    },
}

## `build_disease_manifest`

In [14]:
def build_disease_manifest(crop_name):
    df = pd.read_csv(MANIFEST_PATH)
    df = df[df["crop"] == crop_name].copy()

    crop_map = CANONICAL_DISEASE_MAP.get(crop_name, {})
    df["disease_raw"] = df["disease_raw"].astype(str).str.strip()

    # Exclude pest-damage folders
    before = len(df)
    df = df[~df["disease_raw"].isin(PEST_CLASSES_TO_EXCLUDE)]
    excluded = before - len(df)
    if excluded:
        print(f"[{crop_name}] Excluded {excluded} pest-damage images (reserved for YOLOv8 module)")

    # Apply canonical mapping; anything unmapped passes through as-is and
    # gets flagged so you notice it
    unmapped = set(df["disease_raw"]) - set(crop_map.keys())
    if unmapped:
        print(f"[{crop_name}] WARNING: no canonical mapping for: {sorted(unmapped)}"
              f" -- these will be used as their raw folder names. Add them to"
              f" CANONICAL_DISEASE_MAP if that's not what you want.")

    df["disease"] = df["disease_raw"].map(lambda x: crop_map.get(x, x))
    return df

## `prune_rare_classes`

In [15]:
def prune_rare_classes(df, min_count=MIN_CLASS_COUNT):
    counts = df["disease"].value_counts()
    rare = counts[counts < min_count].index.tolist()
    if rare:
        print(f"Dropping rare classes (< {min_count} images): {rare}")
    return df[~df["disease"].isin(rare)]

## `split_and_materialize`

In [16]:
def split_and_materialize(crop_name, df, train_size=0.7, val_size=0.15, test_size=0.15, seed=42):
    import os
    import sys
    locked_train = df[df["environment"] == "derived"]
    splittable = df[df["environment"] != "derived"]

    strat_key = splittable["disease"]
    train_df, temp_df = train_test_split(
        splittable, train_size=train_size, stratify=strat_key, random_state=seed
    )
    train_df = pd.concat([train_df, locked_train], ignore_index=True)

    remaining_key = temp_df["disease"]
    relative_val = val_size / (val_size + test_size)
    val_df, test_df = train_test_split(
        temp_df, train_size=relative_val, stratify=remaining_key, random_state=seed
    )

    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"{crop_name}/{split_name}"):
            dest_dir = OUT_DIR / crop_name.replace(" ", "_") / split_name / row["disease"]
            dest_dir.mkdir(parents=True, exist_ok=True)
            src = Path(row["filepath"])
            dest = dest_dir / f"img_{abs(hash(str(src)))}{src.suffix}"
            
            src_str = str(src.resolve().absolute())
            dest_str = str(dest.resolve().absolute())
            if sys.platform == "win32":
                if not src_str.startswith("\\\\?\\"):
                    src_str = "\\\\?\\" + src_str
                if not dest_str.startswith("\\\\?\\"):
                    dest_str = "\\\\?\\" + dest_str
            shutil.copy2(src_str, dest_str)

    print(f"[{crop_name}] Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

## `prepare_crop_disease_dataset`

In [17]:
def prepare_crop_disease_dataset(crop_name):
    print(f"\n=== {crop_name} ===")
    df = build_disease_manifest(crop_name)
    print(df.groupby("disease").size().sort_values(ascending=False))
    df = prune_rare_classes(df)
    split_and_materialize(crop_name, df)

## Run

In [18]:
for crop in ["Cotton", "Groundnut", "Pepper Bell", "Potato", "Tomato"]:
    prepare_crop_disease_dataset(crop)


=== Cotton ===
[Cotton] Excluded 80 pest-damage images (reserved for YOLOv8 module)
disease
Fusarium Wilt           583
Healthy                 372
Bacterial Blight        293
Verticillium Wilt       265
Alternaria Leaf Spot    172
Curl Virus               85
Target Spot              38
Powdery Mildew           37
dtype: int64


Cotton/test: 100%|██████████| 277/277 [00:02<00:00, 106.01it/s]


[Cotton] Train: 1291  Val: 277  Test: 277

=== Groundnut ===
disease
Leaf Spot               2616
Healthy                 1865
Rust                    1014
Nutrition Deficiency     670
Alternaria Leaf Spot     385
Rosette                   92
dtype: int64


Groundnut/test: 100%|██████████| 695/695 [00:40<00:00, 17.29it/s]


[Groundnut] Train: 5253  Val: 694  Test: 695

=== Pepper Bell ===
[Pepper Bell] Excluded 42 pest-damage images (reserved for YOLOv8 module)
[Pepper Bell] WARNING: no canonical mapping for: ['Mosaic virus'] -- these will be used as their raw folder names. Add them to CANONICAL_DISEASE_MAP if that's not what you want.
disease
Bacterial Spot          3570
Healthy                 1504
Cercospora Leaf Spot    1406
Nutrient Deficiency      388
Leaf Curl                346
Powdery Mildew           182
Edema                     38
Mosaic virus              25
Blossom End Rot           23
Burn                       5
dtype: int64
Dropping rare classes (< 30 images): ['Mosaic virus', 'Blossom End Rot', 'Burn']


...

[Pepper Bell] Train: 5203  Val: 1115  Test: 1116

=== Potato ===
disease
Late Blight     1932
Early Blight    1770
Healthy         1494
Fungi            730
Pest             597
Bacteria         563
Virus            521
Nematode          68
dtype: int64


Potato/test: 100%|██████████| 1152/1152 [00:50<00:00, 22.91it/s]


[Potato] Train: 5372  Val: 1151  Test: 1152

=== Tomato ===
[Tomato] Excluded 423 pest-damage images (reserved for YOLOv8 module)
disease
Early Blight                              1463
Healthy                                   1369
Bacterial Spot                            1215
Late Blight                               1105
Mold Leaf                                  502
Septoria                                   299
Yellow Curl Virus                          144
Mosaic Virus                                90
Burn Leaf                                   16
Deficiency of Phosphorus and Magnesium      13
Wealth Leaf                                  4
dtype: int64
Dropping rare classes (< 30 images): ['Burn Leaf', 'Deficiency of Phosphorus and Magnesium', 'Wealth Leaf']


Tomato/test: 100%|██████████| 929/929 [00:32<00:00, 28.44it/s]

[Tomato] Train: 4330  Val: 928  Test: 929
